# Paper 1 — Golden Age Semantic Reconfiguration

**Working title:** *Reconfiguring the Golden Age: Semantic Networks and the Renaissance–Baroque Transition in Spanish Poetry*

> **Current stage: Phase 5 — Historical Chronology Expansion.**

Phase 4 established a clean poem backbone and a first primary chronology for Garcilaso and Góngora. Phase 5 has four goals:

1. recover additional Góngora links without lowering the original full-text threshold mechanically;
2. turn the still-undated Garcilaso sonnets into an explicit scholarly worklist;
3. reconstruct Herrera's 1582 / 1619 textual-circulation layers without confusing them with composition time;
4. add a **sensitivity-only** Boscán interval and a source-based historical anchor inventory for the remaining Priority-A authors.

We still **do not build semantic networks or choose temporal windows**. Primary composition time, circulation/attestation time, and broad sensitivity envelopes remain separate variables.


In [10]:
import sys, re, shutil, subprocess, unicodedata
from pathlib import Path
from difflib import SequenceMatcher
from collections import Counter, defaultdict

import pandas as pd
import xml.etree.ElementTree as ET

SOURCES = {
    "navarro_tei": (
        "https://github.com/bncolorado/CorpusSonetosSigloDeOro.git",
        "092a5fe70a4065a4d84bfed288bffd3851348f9c",
    ),
    "gongora_scholarly": (
        "https://github.com/gongoradigital/gongoraobra.git",
        "3beadeecc059a7cc48499dc2683bb378a2630978",
    ),
    "herrera_stylistics": (
        "https://github.com/lamusadecima/Digital-Stylistics-Applied-to-Golden-Age.git",
        "0de990eac908897b5e931aeb5c496170ccf35bab",
    ),
}

ROOT = Path("/content/gasr_phase5_sources")
ROOT.mkdir(exist_ok=True)

def clone(name, url, commit):
    dst = ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(["git","clone","--quiet",url,str(dst)], check=True)
    subprocess.run(["git","-C",str(dst),"checkout","--quiet",commit], check=True)
    got = subprocess.check_output(
        ["git","-C",str(dst),"rev-parse","HEAD"], text=True
    ).strip()
    assert got == commit, (name, got, commit)
    return dst

paths = {k: clone(k, *v) for k,v in SOURCES.items()}
N = paths["navarro_tei"]
G = paths["gongora_scholarly"]
HS = paths["herrera_stylistics"]
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"

def local(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def el_text(el):
    return "" if el is None else " ".join(" ".join(el.itertext()).split())

def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", s.lower())

def years_1580_1626(s):
    return sorted(set(
        int(x) for x in re.findall(r"(?<!\d)(1[56]\d{2})(?!\d)", str(s))
        if 1580 <= int(x) <= 1626
    ))

def split_nonempty_lines(text):
    return [x.strip() for x in str(text).splitlines() if x.strip()]

def segment_blank_blocks(path):
    raw = Path(path).read_text(
        encoding="utf-8", errors="replace"
    ).replace("\r\n","\n")
    blocks = re.split(r"\n\s*\n+", raw)
    rows=[]
    for i,b in enumerate(blocks, start=1):
        ls = split_nonempty_lines(b)
        if not ls:
            continue
        rows.append({
            "block_id": i,
            "n_lines": len(ls),
            "text": "\n".join(ls),
            "signature": norm("\n".join(ls)),
            "first_line": ls[0],
            "first2_signature": norm("\n".join(ls[:2])),
        })
    return pd.DataFrame(rows)

print("Pinned sources ready")
for k,v in SOURCES.items():
    print(f"  {k}: {v[1]}")
print("Python", sys.version.split()[0], "| pandas", pd.__version__)


Pinned sources ready
  navarro_tei: 092a5fe70a4065a4d84bfed288bffd3851348f9c
  gongora_scholarly: 3beadeecc059a7cc48499dc2683bb378a2630978
  herrera_stylistics: 0de990eac908897b5e931aeb5c496170ccf35bab
Python 3.13.15 | pandas 2.2.3


## 01. Rebuild the Navarro poem backbone

The TEI parser continues to use the **body-level poem title**, never the generic `<teiHeader>` title. Each Navarro record receives stable text signatures plus first-line / first-two-line signatures that will be used for conservative variant diagnostics.

No bibliographic, witness, or publication date is promoted to composition time.


In [11]:
def body_poem_title(root):
    for body in root.iter():
        if local(body.tag) == "body":
            for x in body.iter():
                if local(x.tag) == "title":
                    t = el_text(x)
                    if t:
                        return t
            break
    return ""

rows, parse_errors = [], []
for fp in sorted(N.rglob("*.xml")):
    try:
        root = ET.parse(fp).getroot()
    except Exception as e:
        parse_errors.append((str(fp), repr(e)))
        continue

    lines = [el_text(x) for x in root.iter() if local(x.tag) == "l"]
    lines = [x for x in lines if x]
    if not lines:
        continue

    author_dir = fp.parent.name
    text = "\n".join(lines)
    bibls = [
        el_text(x) for x in root.iter()
        if local(x.tag) in {"bibl","witness"} and el_text(x)
    ]

    rows.append({
        "n_id": f"{author_dir}::{fp.name}",
        "author_dir": author_dir,
        "title": body_poem_title(root),
        "n_lines": len(lines),
        "text": text,
        "signature": norm(text),
        "first_line": lines[0],
        "first_line_sig": norm(lines[0]),
        "first2_signature": norm("\n".join(lines[:2])),
        "source_bibl": " | ".join(bibls[:4]),
        "source_file": str(fp.relative_to(N)),
    })

n = pd.DataFrame(rows)
assert not parse_errors, parse_errors[:5]
assert len(n) == 5078, f"Expected 5,078 Navarro records, found {len(n)}"

priority_A = {
    "GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa",
    "JuanDeArguijo","JuanDeJauregui","LuisCarrilloySotomayor","Cervantes",
    "Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"
}

print(f"Navarro poems: {len(n):,} | author folders: {n.author_dir.nunique():,}")
display(
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size()
    .sort_values(ascending=False)
    .rename("poems").reset_index()
)


Navarro poems: 5,078 | author folders: 53


,author_dir,poems
0,LopeDeVega_1,699
1,LopeDeVega_2,647
2,Quevedo,517
3,FernandoDeHerrera,320
4,Gongora,115
5,JuanBoscan,100
6,Cervantes,77
7,JuanDeArguijo,70
8,LuisCarrilloySotomayor,50
9,GarcilasoDeLaVega,38


## 02. Temporal schema: keep three clocks separate

The notebook maintains three different kinds of historical evidence:

- `composition_min/max`: **primary** poem chronology;
- `circulation_year`: publication / manuscript / textual-layer attestation;
- `sensitivity_min/max`: broad fallback interval used only in uncertainty analyses.

A broad sensitivity interval is never counted as primary dating.


In [12]:
temporal = n[
    ["n_id","author_dir","title","source_file","first_line"]
].copy()

for c in [
    "composition_min","composition_max","circulation_year",
    "sensitivity_min","sensitivity_max"
]:
    temporal[c] = pd.NA

temporal["temporal_confidence"] = "unassigned"
temporal["temporal_basis"] = ""
temporal["temporal_source"] = ""
temporal["chronology_status"] = "undated"

temporal["circulation_basis"] = ""
temporal["circulation_source"] = ""

temporal["sensitivity_basis"] = ""
temporal["sensitivity_source"] = ""
temporal["sensitivity_status"] = "none"

def assign_primary(n_ids, lo, hi, confidence, basis, source):
    ids = set(n_ids)
    mask = temporal.n_id.isin(ids)
    if mask.sum() != len(ids):
        missing = ids - set(temporal.loc[mask,"n_id"])
        raise ValueError(f"Primary IDs not found: {sorted(missing)[:5]}")
    if not (temporal.loc[mask,"chronology_status"] == "undated").all():
        overlap = temporal.loc[
            mask & temporal.chronology_status.ne("undated"), "n_id"
        ].tolist()
        raise ValueError(f"Primary assignment collision: {overlap[:5]}")
    temporal.loc[mask,"composition_min"] = int(lo)
    temporal.loc[mask,"composition_max"] = int(hi)
    temporal.loc[mask,"temporal_confidence"] = confidence
    temporal.loc[mask,"temporal_basis"] = basis
    temporal.loc[mask,"temporal_source"] = source
    temporal.loc[mask,"chronology_status"] = "primary_dated"

def assign_sensitivity(n_ids, lo, hi, basis, source):
    ids = set(n_ids)
    mask = temporal.n_id.isin(ids)
    if mask.sum() != len(ids):
        missing = ids - set(temporal.loc[mask,"n_id"])
        raise ValueError(f"Sensitivity IDs not found: {sorted(missing)[:5]}")
    temporal.loc[mask,"sensitivity_min"] = int(lo)
    temporal.loc[mask,"sensitivity_max"] = int(hi)
    temporal.loc[mask,"sensitivity_basis"] = basis
    temporal.loc[mask,"sensitivity_source"] = source
    temporal.loc[mask,"sensitivity_status"] = "sensitivity_only"

def set_circulation(n_ids, year, basis, source, keep_earliest=True):
    ids = set(n_ids)
    mask = temporal.n_id.isin(ids)
    if mask.sum() != len(ids):
        missing = ids - set(temporal.loc[mask,"n_id"])
        raise ValueError(f"Circulation IDs not found: {sorted(missing)[:5]}")
    for idx in temporal.index[mask]:
        current = temporal.at[idx,"circulation_year"]
        if pd.isna(current) or (keep_earliest and int(year) < int(current)):
            temporal.at[idx,"circulation_year"] = int(year)
            temporal.at[idx,"circulation_basis"] = basis
            temporal.at[idx,"circulation_source"] = source

print("Temporal schema initialized:", len(temporal), "poems")


Temporal schema initialized: 5078 poems


## 03. Reproduce the Phase 4 primary chronology

### 03.1 Góngora — full-text linkage to the Cátedra Góngora chronology

The first pass remains intentionally strict:

- exact normalized full-text match, or
- full-text similarity ≥ 0.98,
- one-to-one scholarly target,
- unique scholarly chronology year.

This recreates the Phase 4 baseline before any Phase 5 recovery is attempted.


In [13]:
gfile = G / "gongora_obra-poetica.xml"
assert gfile.exists(), gfile
groot = ET.parse(gfile).getroot()
parent = {child: par for par in groot.iter() for child in par}

poem_divs = []
for el in groot.iter():
    xid = el.attrib.get(XML_ID,"")
    if local(el.tag) == "div" and xid.lower().startswith("poem"):
        ls = [x for x in el.iter() if local(x.tag) == "l"]
        if ls:
            poem_divs.append(el)

def shallow_years(el):
    vals = list(el.attrib.values())
    if el.text:
        vals.append(el.text)
    for ch in list(el):
        if local(ch.tag) in {"head","date","label"}:
            vals.append(el_text(ch))
        if ch.tail:
            vals.append(ch.tail)
    ys=[]
    for v in vals:
        ys.extend(years_1580_1626(v))
    return ys

def ancestor_years(el, max_steps=6):
    ys=[]; cur=el
    for _ in range(max_steps):
        ys.extend(shallow_years(cur))
        cur=parent.get(cur)
        if cur is None:
            break
    return sorted(set(ys))

grows=[]
for el in poem_divs:
    ls=[el_text(x) for x in el.iter() if local(x.tag)=="l"]
    ls=[x for x in ls if x]
    ys=ancestor_years(el)
    text="\n".join(ls)
    grows.append({
        "g_id": el.attrib.get(XML_ID,""),
        "g_n": el.attrib.get("n",""),
        "n_lines": len(ls),
        "text": text,
        "signature": norm(text),
        "first_line": ls[0] if ls else "",
        "first_line_sig": norm(ls[0]) if ls else "",
        "first2_signature": norm("\n".join(ls[:2])),
        "year_candidates": ";".join(map(str,ys)),
        "scholarly_year": ys[0] if len(ys)==1 else pd.NA,
        "year_status": "unique" if len(ys)==1 else (
            "ambiguous" if len(ys)>1 else "missing"
        ),
    })

g=pd.DataFrame(grows)
g14=g[g.n_lines.eq(14) & g.signature.ne("")].copy()

print(f"Góngora scholarly poem divisions: {len(g):,}")
display(g.year_status.value_counts().rename_axis("status").reset_index(name="poems"))
print("14-line scholarly poems:", int((g.n_lines==14).sum()))
print(
    "Unique-year range:",
    g.scholarly_year.dropna().min(), "–", g.scholarly_year.dropna().max()
)


Góngora scholarly poem divisions: 481


,status,poems
0,unique,471
1,missing,9
2,ambiguous,1


14-line scholarly poems: 192
Unique-year range: 1580 – 1626


In [14]:
ng = n[n.author_dir.eq("Gongora")].copy()
sig_to_gids = g14.groupby("signature").g_id.apply(list).to_dict()
g_by_id = g.set_index("g_id", drop=False)

link_rows=[]
for r in ng.itertuples(index=False):
    exact_ids = sig_to_gids.get(r.signature, [])
    if len(exact_ids)==1:
        gid=exact_ids[0]
        score=1.0
        method="exact"
    else:
        best_gid=None
        best_score=-1.0
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None, r.signature, gr.signature).ratio()
            if sc > best_score:
                best_gid=gr.g_id
                best_score=sc
        gid=best_gid
        score=best_score
        method="fuzzy_provisional"

    link_rows.append({
        "n_id":r.n_id,
        "g_id":gid,
        "method":method,
        "score":float(score),
        "preaccept": (method=="exact") or (score>=0.98),
    })

glink=pd.DataFrame(link_rows)
pre=glink[glink.preaccept].copy()
collision_ids=set(
    pre.g_id.value_counts()[lambda s:s>1].index
)
glink["accept_phase4"] = (
    glink.preaccept &
    ~glink.g_id.isin(collision_ids)
)

accepted4=glink[glink.accept_phase4].copy()
accepted4=accepted4.merge(
    g[["g_id","scholarly_year","year_status"]],
    on="g_id", how="left"
)
accepted4_dated=accepted4[
    accepted4.year_status.eq("unique") & accepted4.scholarly_year.notna()
].copy()

for r in accepted4_dated.itertuples(index=False):
    assign_primary(
        [r.n_id], int(r.scholarly_year), int(r.scholarly_year),
        "B", "scholarly_chronology_year",
        "Cátedra Góngora / gongoradigital/gongoraobra; "
        "Carreira/Biblioteca Castro chronology"
    )

print("Phase 4 Góngora baseline")
print("  Navarro sonnets:", len(ng))
print("  exact links:", int((glink.method=="exact").sum()))
print("  fuzzy >= .98 candidates:", int(
    ((glink.method=="fuzzy_provisional") & glink.preaccept).sum()
))
print("  accepted one-to-one links:", int(glink.accept_phase4.sum()))
print("  accepted with unique scholarly year:", len(accepted4_dated))
print("  colliding scholarly targets excluded:", len(collision_ids))


Phase 4 Góngora baseline
  Navarro sonnets: 115
  exact links: 12
  fuzzy >= .98 candidates: 41
  accepted one-to-one links: 53
  accepted with unique scholarly year: 53
  colliding scholarly targets excluded: 0


### 03.2 Garcilaso — conservative Lapesa/Rivers seed

The numbering is cross-checked independently from the body-level Roman numeral title and from the XML filename. Only the same conservative 18-sonnet seed used in Phase 4 is promoted to the primary axis.


In [15]:
roman_vals={"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}
def roman_to_int(s):
    total=0; prev=0
    for ch in reversed(str(s).upper()):
        v=roman_vals.get(ch,0)
        total += -v if v<prev else v
        prev=max(prev,v)
    return total

gar=n[n.author_dir.eq("GarcilasoDeLaVega")].copy()
gar["roman_token"]=gar.title.str.extract(
    r"^\s*-\s*([IVXLCDM]+)\s*-\s*$", expand=False
)
gar["title_no"]=gar.roman_token.map(
    lambda x: roman_to_int(x) if isinstance(x,str) and x else pd.NA
).astype("Int64")
gar["file_no"]=pd.to_numeric(
    gar.n_id.str.extract(r"_(\d+)\.xml$",expand=False),
    errors="coerce"
).astype("Int64")
gar["sonnet_no"]=gar.file_no

assert len(gar)==38
assert gar.file_no.notna().all()
assert gar.file_no.nunique()==38
title_parsed=int(gar.title_no.notna().sum())
agreements=int((gar.title_no==gar.file_no).fillna(False).sum())

print("Garcilaso numbering:")
print("  Roman-title parsed:", title_parsed)
print("  filename numbers parsed:", int(gar.file_no.notna().sum()))
print("  title/filename agreements:", agreements)
if title_parsed==38 and agreements==38:
    print("  integrity: PASSED (38/38)")
else:
    print("  WARNING: title cross-check incomplete; filename numbering remains canonical.")

GAR_CHRONOLOGY = {
    **{i:(1526,1532,"B","scholarly_phase_interval") for i in [1,2,3,4,6,26,27]},
    25:(1534,1535,"B","scholarly_interval"),
    33:(1535,1535,"A","historically_anchored_scholarly_year"),
    35:(1535,1535,"A","historically_anchored_scholarly_year"),
    **{i:(1533,1535,"B","revised_scholarly_interval") for i in [7,8,12,15,19,28,30,31]},
}

gassign=[]
for no,(lo,hi,conf,basis) in GAR_CHRONOLOGY.items():
    z=gar[gar.sonnet_no.eq(no)]
    if len(z)!=1:
        raise ValueError(f"Garcilaso canonical number {no} resolved to {len(z)} records")
    nid=z.iloc[0].n_id
    assign_primary(
        [nid],lo,hi,conf,basis,
        "Rafael Lapesa chronology as summarized/discussed by "
        "E. L. Rivers (Centro Virtual Cervantes) plus AISO/AISPI checks"
    )
    gassign.append({
        "sonnet_no":no,"n_id":nid,
        "composition_min":lo,"composition_max":hi,
        "temporal_confidence":conf,"temporal_basis":basis
    })

gar_chron=pd.DataFrame(gassign).sort_values("sonnet_no")
print("Garcilaso primary-dated:",len(gar_chron),"/",len(gar))


Garcilaso numbering:
  Roman-title parsed: 38
  filename numbers parsed: 38
  title/filename agreements: 38
  integrity: PASSED (38/38)
Garcilaso primary-dated: 18 / 38


## 04. Phase 5A — Recover Góngora variants without mechanically lowering 0.98

The 62 Phase-4 non-links are examined with a **second, orthogonal identifier**: the first two verse lines.

A Phase-5 recovery is accepted only when all of the following hold:

1. the first-two-line normalized signature occurs in exactly one 14-line scholarly poem;
2. full-text similarity is at least 0.95;
3. the scholarly target has a unique chronology year;
4. that scholarly target was not already used in Phase 4;
5. no two new Navarro poems compete for the same scholarly target.

This is stricter than simply lowering the full-text threshold. All other cases remain a review worklist.


In [23]:
phase4_nids=set(glink.loc[glink.accept_phase4,"n_id"])
phase4_gids=set(glink.loc[glink.accept_phase4,"g_id"])
unmatched=ng[~ng.n_id.isin(phase4_nids)].copy()

first2_index=g14.groupby("first2_signature").g_id.apply(list).to_dict()
first1_index=g14.groupby("first_line_sig").g_id.apply(list).to_dict()

diag=[]
for r in unmatched.itertuples(index=False):
    ids2=first2_index.get(r.first2_signature,[])
    ids1=first1_index.get(r.first_line_sig,[])

    best14_gid=None; best14_score=-1.0
    for gr in g14.itertuples(index=False):
        sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
        if sc>best14_score:
            best14_gid=gr.g_id; best14_score=sc

    bestall_gid=None; bestall_score=-1.0
    for gr in g.itertuples(index=False):
        if not gr.signature:
            continue
        sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
        if sc>bestall_score:
            bestall_gid=gr.g_id; bestall_score=sc

    unique2_gid=ids2[0] if len(ids2)==1 else None
    unique1_gid=ids1[0] if len(ids1)==1 else None

    score2=pd.NA
    year2=pd.NA
    year2_status=""
    if unique2_gid is not None:
        gr=g_by_id.loc[unique2_gid]
        score2=SequenceMatcher(None,r.signature,gr.signature).ratio()
        year2=gr.scholarly_year
        year2_status=gr.year_status

    bestall_lines=int(g_by_id.loc[bestall_gid].n_lines) if bestall_gid else pd.NA

    if unique2_gid is not None and float(score2)>=0.95:
        dclass="unique_first2_strong_variant"
    elif unique1_gid is not None and best14_score>=0.95:
        dclass="unique_incipit_strong_variant"
    elif best14_score>=0.95:
        dclass="strong_fulltext_variant"
    elif best14_score>=0.90:
        dclass="review_90_95"
    elif (
        bestall_gid is not None and bestall_lines!=14
        and bestall_score-best14_score>=0.05
    ):
        dclass="best_target_non14"
    else:
        dclass="low_similarity_or_different_witness"

    diag.append({
        "n_id":r.n_id,
        "title":r.title,
        "first_line":r.first_line,
        "unique_first2_gid":unique2_gid,
        "unique_first2_score":score2,
        "unique_first2_year":year2,
        "unique_first2_year_status":year2_status,
        "unique_first1_gid":unique1_gid,
        "best14_gid":best14_gid,
        "best14_score":best14_score,
        "bestall_gid":bestall_gid,
        "bestall_score":bestall_score,
        "bestall_n_lines":bestall_lines,
        "diagnostic_class":dclass,
    })

gdiag=pd.DataFrame(diag)

gdiag["phase5_preaccept"]=(
    gdiag.unique_first2_gid.notna()
    & pd.to_numeric(gdiag.unique_first2_score,errors="coerce").ge(0.95)
    & gdiag.unique_first2_year.notna()
    & gdiag.unique_first2_year_status.eq("unique")
    & ~gdiag.unique_first2_gid.isin(phase4_gids)
)

new_collisions=set(
    gdiag.loc[gdiag.phase5_preaccept,"unique_first2_gid"]
    .value_counts()[lambda s:s>1].index
)
gdiag["phase5_accept"]=(
    gdiag.phase5_preaccept
    & ~gdiag.unique_first2_gid.isin(new_collisions)
)

new_gongora=gdiag[gdiag.phase5_accept].copy()
for r in new_gongora.itertuples(index=False):
    assign_primary(
        [r.n_id], int(r.unique_first2_year), int(r.unique_first2_year),
        "B", "scholarly_chronology_year_variant_link",
        "Cátedra Góngora chronology; unique first-two-line signature + "
        ">=0.95 full-text similarity; one-to-one target"
    )

print("Phase 5 Góngora diagnostic")
print("  Phase-4 unmatched:",len(unmatched))
print("  Phase-5 recovered:",len(new_gongora))
print("  new target collisions rejected:",len(new_collisions))
print("  total Góngora primary-dated:",
      int((temporal.author_dir.eq("Gongora") &
           temporal.chronology_status.eq("primary_dated")).sum()))
print()
display(
    gdiag.diagnostic_class.value_counts()
    .rename_axis("diagnostic_class").reset_index(name="poems")
)
print("Accepted Phase-5 recoveries:")
display(
    new_gongora[
        ["n_id","first_line","unique_first2_gid",
         "unique_first2_score","unique_first2_year"]
    ].sort_values("unique_first2_year")
)


ValueError: Primary assignment collision: ['Gongora::Gongora_31.xml']

## 05. Phase 5B — Garcilaso undated worklist

The remaining Garcilaso sonnets are **not filled with interpolated years**. We export a compact worklist with canonical sonnet number and incipit so that the next scholarly pass can search poem-specific evidence.

This preserves the methodological advantage of uncertainty rather than manufacturing precision.


In [17]:
dated_gar=set(gar_chron.n_id)
gar_worklist=(
    gar[~gar.n_id.isin(dated_gar)]
    [["sonnet_no","n_id","title","first_line","source_bibl"]]
    .sort_values("sonnet_no")
    .reset_index(drop=True)
)
gar_worklist["next_evidence_needed"] = (
    "poem-specific scholarly chronology / historical anchor; "
    "otherwise remain undated"
)

print("Garcilaso still undated:",len(gar_worklist))
display(gar_worklist)


Garcilaso still undated: 20


,sonnet_no,n_id,title,first_line,source_bibl,next_evidence_needed
0,5,GarcilasoDeLaVega::GarcilasoDeLaVega_05.xml,-V-,"Escrito está en mi alma vuestro gesto,",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
1,9,GarcilasoDeLaVega::GarcilasoDeLaVega_09.xml,-IX-,"Señora mia, si yo de vos ausente",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
2,10,GarcilasoDeLaVega::GarcilasoDeLaVega_10.xml,-X-,"¡Oh dulces prendas por mi mal halladas,",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
3,11,GarcilasoDeLaVega::GarcilasoDeLaVega_11.xml,-XI-,"Hermosas ninfas, que en el rio metidas",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
4,13,GarcilasoDeLaVega::GarcilasoDeLaVega_13.xml,-XIII-,"A Dafne ya los brazos le crecían,",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
5,14,GarcilasoDeLaVega::GarcilasoDeLaVega_14.xml,-XIV-,Como la tierna madre que el doliente,Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
6,16,GarcilasoDeLaVega::GarcilasoDeLaVega_16.xml,-XVI-,"No las francesas armas odïosas,",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
7,17,GarcilasoDeLaVega::GarcilasoDeLaVega_17.xml,-XVII-,"Pensando que el camino iba derecho,",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
8,18,GarcilasoDeLaVega::GarcilasoDeLaVega_18.xml,-XVIII-,"Si a vuestra voluntad yo soy de cera,",Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...
9,20,GarcilasoDeLaVega::GarcilasoDeLaVega_20.xml,-XX-,Con tal fuerza y vigor son concertados,Sonetos de Garcilaso de La Vega . Biblioteca V...,poem-specific scholarly chronology / historica...


## 06. Phase 5C — Herrera textual layers: 1582 vs 1619 as circulation, not composition

The Hernández-Lorenzo companion repository provides `H.txt` and `P2.txt`. We compare its 14-line blocks against the 320 Navarro Herrera sonnets.

The goal is **not** to date composition. Instead we record the earliest identifiable textual-circulation layer:

- exact H-layer match → 1582 circulation/edition evidence;
- P2-only exact match → 1619 posthumous circulation/edition evidence;
- both → earliest attested layer 1582;
- neither → unresolved textual witness / variant.

These fields are kept outside the primary composition axis.


In [18]:
U = HS / "corpus" / "untagged_corpus"
Hfile = U / "H.txt"
P2file = U / "P2.txt"
Bfile = U / "JuanBoscan_Sonetos.txt"
assert Hfile.exists() and P2file.exists() and Bfile.exists()

Hblocks=segment_blank_blocks(Hfile)
P2blocks=segment_blank_blocks(P2file)
H14=Hblocks[Hblocks.n_lines.eq(14)].copy()
P214=P2blocks[P2blocks.n_lines.eq(14)].copy()

nh=n[n.author_dir.eq("FernandoDeHerrera")].copy()
Hsig=set(H14.signature)
P2sig=set(P214.signature)

herrera_map=nh[
    ["n_id","title","first_line","signature","source_bibl"]
].copy()
herrera_map["H_exact"]=herrera_map.signature.isin(Hsig)
herrera_map["P2_exact"]=herrera_map.signature.isin(P2sig)

def hclass(r):
    if r.H_exact and r.P2_exact:
        return "H_and_P2_exact"
    if r.H_exact:
        return "H_exact_only"
    if r.P2_exact:
        return "P2_exact_only"
    return "unresolved_variant_or_other"

herrera_map["layer_class"]=herrera_map.apply(hclass,axis=1)

h_ids=herrera_map.loc[herrera_map.H_exact,"n_id"].tolist()
p2_only_ids=herrera_map.loc[
    herrera_map.P2_exact & ~herrera_map.H_exact,"n_id"
].tolist()

if h_ids:
    set_circulation(
        h_ids,1582,"H / Algunas obras textual layer",
        "Hernández-Lorenzo companion corpus; Algunas obras (1582)"
    )
if p2_only_ids:
    set_circulation(
        p2_only_ids,1619,"P2 / Versos posthumous textual layer",
        "Hernández-Lorenzo companion corpus; Versos (1619)"
    )

print("Herrera layer audit")
print("  Navarro Herrera sonnets:",len(nh))
print("  H source blocks:",len(Hblocks),"| 14-line:",len(H14))
print("  P2 source blocks:",len(P2blocks),"| 14-line:",len(P214))
display(
    herrera_map.layer_class.value_counts()
    .rename_axis("layer_class").reset_index(name="poems")
)
print(
    "Primary composition dates assigned from H/P2:",
    int((temporal.author_dir.eq("FernandoDeHerrera") &
         temporal.chronology_status.eq("primary_dated")).sum()),
    "(must remain 0 in this phase)"
)


Herrera layer audit
  Navarro Herrera sonnets: 320
  H source blocks: 78 | 14-line: 78
  P2 source blocks: 200 | 14-line: 172


,layer_class,poems
0,unresolved_variant_or_other,231
1,P2_exact_only,63
2,H_exact_only,26


Primary composition dates assigned from H/P2: 0 (must remain 0 in this phase)


## 07. Phase 5D — Boscán: broad sensitivity interval, not primary dating

Boscán's meeting with Andrea Navagero in Granada in **1526** is the standard historical anchor for his systematic cultivation of Italianate forms; he died in **1542**, and *Las obras de Boscán y algunas de Garcilaso* appeared posthumously in 1543.

For the sonnet corpus, `[1526, 1542]` is therefore introduced only as a **confidence-C sensitivity envelope**. It is *not* counted as poem-level chronology, because internal ordering and individual composition dates remain unresolved.

We also audit the Hernández-Lorenzo Boscán text layer against Navarro to expose any corpus-count discrepancy before later promotion.


In [19]:
Bblocks=segment_blank_blocks(Bfile)
B14=Bblocks[Bblocks.n_lines.eq(14)].copy()
nb=n[n.author_dir.eq("JuanBoscan")].copy()

Bsig=set(B14.signature)
boscan_map=nb[
    ["n_id","title","first_line","signature","source_bibl"]
].copy()
boscan_map["external_layer_exact"]=boscan_map.signature.isin(Bsig)

assign_sensitivity(
    nb.n_id.tolist(),1526,1542,
    "Boscán Italianate-sonnet activity envelope; sensitivity only",
    "Navagero-Boscán Granada encounter (1526) to Boscán's death (1542); "
    "CVC/Biblioteca Virtual Miguel de Cervantes"
)

print("Boscán layer audit")
print("  Navarro Boscán sonnets:",len(nb))
print("  companion source blocks:",len(Bblocks),"| 14-line:",len(B14))
print("  exact layer matches:",int(boscan_map.external_layer_exact.sum()))
print("  sensitivity interval assigned:",1526,"–",1542)
print("  PRIMARY Boscán dates:",
      int((temporal.author_dir.eq("JuanBoscan") &
           temporal.chronology_status.eq("primary_dated")).sum()),
      "(must remain 0)")


Boscán layer audit
  Navarro Boscán sonnets: 100
  companion source blocks: 100 | 14-line: 99
  exact layer matches: 96
  sensitivity interval assigned: 1526 – 1542
  PRIMARY Boscán dates: 0 (must remain 0)


## 08. Scholarly anchor inventory for the remaining Priority-A authors

This table records **what historical evidence exists** before we attempt poem-level dating. It deliberately distinguishes:

- `primary_chronology`: admissible for `composition_min/max`;
- `sensitivity_envelope`: usable only in uncertainty analyses;
- `circulation_or_attestation`: useful historical evidence but not composition time;
- `needs_poem_level_reconstruction`: no safe automatic assignment yet.

The notebook does not scrape these pages at runtime; URLs are recorded for provenance and manual scholarly review.


In [20]:
anchor_rows = [
    {
        "author_dir":"GarcilasoDeLaVega",
        "anchor":"Lapesa/Rivers scholarly sonnet chronology",
        "year_or_interval":"poem-specific / intervals",
        "evidence_class":"primary_chronology",
        "automatic_action":"18-sonnet conservative seed already assigned",
        "source":"https://cvc.cervantes.es/actcult/garcilaso/anotaciones/rivers.htm",
    },
    {
        "author_dir":"JuanBoscan",
        "anchor":"Navagero encounter; systematic Italianate experiment",
        "year_or_interval":"1526–1542",
        "evidence_class":"sensitivity_envelope",
        "automatic_action":"sensitivity only; no primary assignment",
        "source":"https://www.cervantesvirtual.com/obra-visor/sonetos--35/",
    },
    {
        "author_dir":"FernandoDeHerrera",
        "anchor":"earliest documented poetic preliminaries; 1578 manuscript; Algunas obras; Versos",
        "year_or_interval":"1561–1565 / 1578 / 1582 / 1619",
        "evidence_class":"circulation_or_attestation",
        "automatic_action":"H/P2 circulation layers only",
        "source":"https://www.cervantesvirtual.com/portales/fernando_de_herrera/autor_cronologia/",
    },
    {
        "author_dir":"Gongora",
        "anchor":"Cátedra Góngora / Carreira chronology",
        "year_or_interval":"1580–1626",
        "evidence_class":"primary_chronology",
        "automatic_action":"text-linked scholarly years",
        "source":"https://www.uco.es/catedragongora/?page_id=3483",
    },
    {
        "author_dir":"PedroEspinosa",
        "anchor":"Primera parte de Flores de poetas ilustres; 19 own poems",
        "year_or_interval":"1605 (dedication 1603)",
        "evidence_class":"circulation_or_attestation",
        "automatic_action":"no primary assignment until poem identities are linked",
        "source":"https://cervantes.bne.es/es/su-biblioteca/espinosa-pedro-primera-parte-flores-poetas-ilustres-espana-",
    },
    {
        "author_dir":"JuanDeArguijo",
        "anchor":"six compositions in Flores; later sonnet manuscript tradition",
        "year_or_interval":"1605 / early 17th c.",
        "evidence_class":"circulation_or_attestation",
        "automatic_action":"no primary assignment until individual poems are identified",
        "source":"https://www.classicahispalensia.es/estudios/58-juan-de-arguijo-y-la-sevilla-del-siglo-de-oro-el-contexto-literario",
    },
    {
        "author_dir":"JuanDeJauregui",
        "anchor":"Rimas de Don Iuan de Iauregui",
        "year_or_interval":"1618",
        "evidence_class":"circulation_or_attestation",
        "automatic_action":"no composition assignment",
        "source":"https://www.cervantesvirtual.com/portales/portal_nacional_venezuela/obras/autor/jauregui-juan-de-38294",
    },
    {
        "author_dir":"LuisCarrilloySotomayor",
        "anchor":"Obras posthumous edition",
        "year_or_interval":"1611",
        "evidence_class":"circulation_or_attestation",
        "automatic_action":"no composition assignment",
        "source":"https://cvc.cervantes.es/",
    },
    {
        "author_dir":"Cervantes",
        "anchor":"heterogeneous occasional / paratextual sonnet chronology",
        "year_or_interval":"requires poem-level reconstruction",
        "evidence_class":"needs_poem_level_reconstruction",
        "automatic_action":"worklist only",
        "source":"https://www.cervantesvirtual.com/",
    },
    {
        "author_dir":"LopeDeVega_1",
        "anchor":"book-level Rimas chronology",
        "year_or_interval":"1602 onward; source-specific",
        "evidence_class":"needs_book_level_reconstruction",
        "automatic_action":"defer until source groups are mapped",
        "source":"https://www.cervantesvirtual.com/",
    },
    {
        "author_dir":"LopeDeVega_2",
        "anchor":"book-level Rimas chronology",
        "year_or_interval":"source-specific",
        "evidence_class":"needs_book_level_reconstruction",
        "automatic_action":"defer until source groups are mapped",
        "source":"https://www.cervantesvirtual.com/",
    },
    {
        "author_dir":"Quevedo",
        "anchor":"early printed poems in Flores (1605) plus later editorial structures",
        "year_or_interval":"1605 attestation for a subset",
        "evidence_class":"needs_poem_level_reconstruction",
        "automatic_action":"do not generalize 1605 to whole corpus",
        "source":"https://www.cervantesvirtual.com/",
    },
]

anchors=pd.DataFrame(anchor_rows)
display(anchors)

assert set(anchors.author_dir)==priority_A, (
    "Priority-A anchor inventory must contain each author group exactly once"
)


,author_dir,anchor,year_or_interval,evidence_class,automatic_action,source
0,GarcilasoDeLaVega,Lapesa/Rivers scholarly sonnet chronology,poem-specific / intervals,primary_chronology,18-sonnet conservative seed already assigned,https://cvc.cervantes.es/actcult/garcilaso/ano...
1,JuanBoscan,Navagero encounter; systematic Italianate expe...,1526–1542,sensitivity_envelope,sensitivity only; no primary assignment,https://www.cervantesvirtual.com/obra-visor/so...
2,FernandoDeHerrera,earliest documented poetic preliminaries; 1578...,1561–1565 / 1578 / 1582 / 1619,circulation_or_attestation,H/P2 circulation layers only,https://www.cervantesvirtual.com/portales/fern...
3,Gongora,Cátedra Góngora / Carreira chronology,1580–1626,primary_chronology,text-linked scholarly years,https://www.uco.es/catedragongora/?page_id=3483
4,PedroEspinosa,Primera parte de Flores de poetas ilustres; 19...,1605 (dedication 1603),circulation_or_attestation,no primary assignment until poem identities ar...,https://cervantes.bne.es/es/su-biblioteca/espi...
5,JuanDeArguijo,six compositions in Flores; later sonnet manus...,1605 / early 17th c.,circulation_or_attestation,no primary assignment until individual poems a...,https://www.classicahispalensia.es/estudios/58...
6,JuanDeJauregui,Rimas de Don Iuan de Iauregui,1618,circulation_or_attestation,no composition assignment,https://www.cervantesvirtual.com/portales/port...
7,LuisCarrilloySotomayor,Obras posthumous edition,1611,circulation_or_attestation,no composition assignment,https://cvc.cervantes.es/
8,Cervantes,heterogeneous occasional / paratextual sonnet ...,requires poem-level reconstruction,needs_poem_level_reconstruction,worklist only,https://www.cervantesvirtual.com/
9,LopeDeVega_1,book-level Rimas chronology,1602 onward; source-specific,needs_book_level_reconstruction,defer until source groups are mapped,https://www.cervantesvirtual.com/


## 09. Phase 5 coverage and integrity audit

Two coverages are reported separately:

1. **primary coverage** — poems that may enter the main diachronic analysis;
2. **sensitivity coverage** — poems with a broad fallback interval but no primary date.

No publication/edition/witness year is allowed to leak into the primary composition axis.


In [21]:
coverage=(
    n[n.author_dir.isin(priority_A)]
    .groupby("author_dir").size().rename("total_poems").to_frame()
    .join(
        temporal[temporal.chronology_status.eq("primary_dated")]
        .groupby("author_dir").size().rename("primary_dated")
    )
    .join(
        temporal[temporal.sensitivity_status.eq("sensitivity_only")]
        .groupby("author_dir").size().rename("sensitivity_only")
    )
    .join(
        temporal[temporal.circulation_year.notna()]
        .groupby("author_dir").size().rename("circulation_attested")
    )
    .fillna(0)
)
for c in ["primary_dated","sensitivity_only","circulation_attested"]:
    coverage[c]=coverage[c].astype(int)

coverage["primary_pct"]=(100*coverage.primary_dated/coverage.total_poems).round(1)
coverage["sensitivity_pct"]=(100*coverage.sensitivity_only/coverage.total_poems).round(1)
coverage=coverage.sort_values(
    ["primary_pct","sensitivity_pct","total_poems"],
    ascending=[False,False,False]
).reset_index()

display(coverage)

primary=temporal[temporal.chronology_status.eq("primary_dated")].copy()
sensitivity=temporal[temporal.sensitivity_status.eq("sensitivity_only")].copy()

assert primary.composition_min.notna().all()
assert primary.composition_max.notna().all()
assert (primary.composition_min.astype(int)<=primary.composition_max.astype(int)).all()
assert not primary.temporal_basis.str.contains(
    "publication|witness|edition|circulation",
    case=False,regex=True
).any(), "Non-composition evidence leaked into primary axis."

assert sensitivity.sensitivity_min.notna().all()
assert sensitivity.sensitivity_max.notna().all()

print("Primary-dated poems:",len(primary))
print("Sensitivity-only poems:",len(sensitivity))
print("Circulation/attestation records:",int(temporal.circulation_year.notna().sum()))
print("Primary confidence distribution:")
display(
    primary.temporal_confidence.value_counts()
    .rename_axis("confidence").reset_index(name="poems")
)
print("TEMPORAL INTEGRITY CHECKS: PASSED")


,author_dir,total_poems,primary_dated,sensitivity_only,circulation_attested,primary_pct,sensitivity_pct
0,Gongora,115,58,0,0,50.4,0.0
1,GarcilasoDeLaVega,38,18,0,0,47.4,0.0
2,JuanBoscan,100,0,100,0,0.0,100.0
3,LopeDeVega_1,699,0,0,0,0.0,0.0
4,LopeDeVega_2,647,0,0,0,0.0,0.0
5,Quevedo,517,0,0,0,0.0,0.0
6,FernandoDeHerrera,320,0,0,89,0.0,0.0
7,Cervantes,77,0,0,0,0.0,0.0
8,JuanDeArguijo,70,0,0,0,0.0,0.0
9,LuisCarrilloySotomayor,50,0,0,0,0.0,0.0


Primary-dated poems: 76
Sensitivity-only poems: 100
Circulation/attestation records: 89
Primary confidence distribution:


,confidence,poems
0,B,74
1,A,2


TEMPORAL INTEGRITY CHECKS: PASSED


## 10. Runtime exports and Phase 5 checkpoint

The derived CSVs remain runtime products for inspection. The notebook plus pinned source commits and `manuscript/chronology_sources.md` are the provenance record.

After this execution, the next decision is empirical: whether the strengthened dated subset is sufficiently distributed across the sixteenth–seventeenth-century transition to support interval/probabilistic chronology, or whether more poem-level scholarly acquisition is required before any network window is chosen.


In [22]:
OUT=Path("/content/gasr_phase5_outputs")
OUT.mkdir(exist_ok=True)

temporal.to_csv(OUT/"temporal_master_phase5.csv",index=False)
glink.to_csv(OUT/"gongora_phase4_links.csv",index=False)
gdiag.to_csv(OUT/"gongora_phase5_recovery_diagnostics.csv",index=False)
gar_chron.to_csv(OUT/"garcilaso_chronology_seed.csv",index=False)
gar_worklist.to_csv(OUT/"garcilaso_unassigned_worklist.csv",index=False)
herrera_map.to_csv(OUT/"herrera_textual_layer_map.csv",index=False)
boscan_map.to_csv(OUT/"boscan_layer_audit.csv",index=False)
anchors.to_csv(OUT/"historical_anchor_inventory.csv",index=False)
coverage.to_csv(OUT/"phase5_author_coverage.csv",index=False)

print("Runtime outputs:")
for p in sorted(OUT.glob("*.csv")):
    print(" ",p)

print()
print("PHASE 5 CHECKPOINT")
print("------------------")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
print("Next: inspect Góngora Phase-5 recoveries, Herrera layer coverage,")
print("Boscán sensitivity coverage, and the remaining poem-level chronology worklist.")


Runtime outputs:
  /content/gasr_phase5_outputs/boscan_layer_audit.csv
  /content/gasr_phase5_outputs/garcilaso_chronology_seed.csv
  /content/gasr_phase5_outputs/garcilaso_unassigned_worklist.csv
  /content/gasr_phase5_outputs/gongora_phase4_links.csv
  /content/gasr_phase5_outputs/gongora_phase5_recovery_diagnostics.csv
  /content/gasr_phase5_outputs/herrera_textual_layer_map.csv
  /content/gasr_phase5_outputs/historical_anchor_inventory.csv
  /content/gasr_phase5_outputs/phase5_author_coverage.csv
  /content/gasr_phase5_outputs/temporal_master_phase5.csv

PHASE 5 CHECKPOINT
------------------
Save this executed notebook to GitHub.
Do NOT build semantic networks yet.
Next: inspect Góngora Phase-5 recoveries, Herrera layer coverage,
Boscán sensitivity coverage, and the remaining poem-level chronology worklist.
